## Fine tuning

In [ ]:
import os
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
from langchain_community.document_loaders import UnstructuredXMLLoader
import xml.etree.ElementTree as ET
from huggingface_hub import login
from trl import SFTTrainer, SFTConfig


# Configuration for environment
os.environ["WANDB_DISABLED"] = "true"

### Configuração geral

In [ ]:
# Configuration
MODEL_ID = "epfl-llm/meditron-7B"  # Foundation model para coisas de saúde/medicina
GITHUB_REPO_ID = "abachaa/MedQuAD" # Indicado pelo professor
DATA_FILE_PATH = "data.jsonl"         # Example filename in the repo

# Hiperparametros (Otimizado para rodar em uma RTX 3060 12GB)
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
LR_SCHEDULER_TYPE = "linear"
MAX_SAMPLES = 1000
MAX_SEQ_LENGTH = 512

# Login to Hugging Face
login()


### Carregando os dados do repositório git

In [ ]:
import subprocess
from pathlib import Path

medquad_dir = Path("MedQuAD")

if not medquad_dir.is_dir():
    print("Pasta MedQuAD não encontrada. Clonando repositório...")
    subprocess.run(["git", "clone", "https://github.com/abachaa/MedQuAD.git"], check=True)
    print("Repositório clonado")
else:
    print("Pasta MedQuAD já existe, não é necessário clonar novamente.")

In [ ]:
def parse_file(file_path):
    tree = ET.parse(file_path)
    qapairs = tree.iter("QAPair")

    parsed_data = []

    for qapair in qapairs:
        question_elem = qapair.find("Question")
        answer_elem = qapair.find("Answer")

        if question_elem is not None and answer_elem is not None:
            question = question_elem.text
            answer = answer_elem.text

            if question is not None and answer is not None:
                parsed_data.append({"question": question, "answer": answer})
    return parsed_data

In [ ]:
docs = []

for folder in os.listdir(medquad_dir):
    folder_path = medquad_dir / folder
    if folder_path.is_dir():
        for file in os.listdir(folder_path):
            if file.endswith(".xml"):
                file_path = folder_path / file
                docs.extend(parse_file(file_path))

df = pd.DataFrame(docs)

In [ ]:
df

### Baixando foundation model do Hugging Face

In [ ]:
# Configuração para quantização 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir="./cache")
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="cuda", cache_dir="./cache", quantization_config=bnb_config)

### Formatando os dados para o fine tuning, usando um template alpaca

In [ ]:
# Definindo o template - template inspirado no template contido no model card do Meditron
prompt_template = """###System:
You are a helpful, respectful, and honest assistant. Always answer as helpfully as possible, while being safe. Your answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your responses are socially unbiased and positive in nature.

If a question does not make any sense, or is not factually coherent, explain why instead of answering something not correct. If you don't know the answer to a question, don't share false information.

### User:
{question}

### Assistant:
{answer}"""

dataset = Dataset.from_pandas(df)

def format_alpaca_prompt(example):
    return {"text": prompt_template.format(question=example['question'], answer=example['answer'] + tokenizer.eos_token)}

dataset = dataset.map(format_alpaca_prompt, remove_columns=["question", "answer"])

dataset

In [ ]:
model = prepare_model_for_kbit_training(model)

In [ ]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], # Standard for many models
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
training_args = SFTConfig(
    output_dir="./meditron-finetuned",
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_steps=5,
    learning_rate=LEARNING_RATE,
    num_train_epochs=3,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    logging_steps=1,
    fp16=False,
    save_strategy="steps",
    save_steps="60",
    report_to="none",
    optim="adamw_8bit",
    seed=74,
    max_length=MAX_SEQ_LENGTH,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=dataset,
)

trainer.train()